In [1]:
"""
Script Name: transform_WB_SE4ALL_data.ipynb
Authores: Walter Arredondo, Carlos Daza, Andrés Coral
Fecha: 2026-10-03
Fase ETL: Transformación
Descripción:
Obtiene el resultado de la API del World Bank Group (base de datos WB_SE4ALL),
guardado en data/raw/energia_sostenible_WB_SE4ALL.csv, realiza las transformaciones
necesarias y las guarda como archivo CSV para posterior carga.
Entradas: - CSV con los resultados de la extracción: data/raw/energia_sostenible_WB_SE4ALL.csv
Salidas: - Archivo CSV con las transformaciones realizadas: data/processed/energia_sostenible_WB_SE4ALL.csv
Notas: - Ejecutar este script una sola vez
"""

'\nScript Name: transform_WB_SE4ALL_data.ipynb\nAuthores: Walter Arredondo, Carlos Daza, Andrés Coral\nFecha: 2026-10-03\nFase ETL: Transformación\nDescripción:\nObtiene el resultado de la API del World Bank Group (base de datos WB_SE4ALL),\nguardado en data/raw/energia_sostenible_WB_SE4ALL.csv, realiza las transformaciones\nnecesarias y las guarda como archivo CSV para posterior carga.\nEntradas: - CSV con los resultados de la extracción: data/raw/energia_sostenible_WB_SE4ALL.csv\nSalidas: - Archivo CSV con las transformaciones realizadas: data/processed/energia_sostenible_WB_SE4ALL.csv\nNotas: - Ejecutar este script una sola vez\n'

# Recuperar información de la extracción

A continuación se carga el csv "data/raw/energia_sostenible_WB_SE4ALL.csv" generado por la fase de extracción.

In [2]:
import pandas as pd

In [3]:
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
ruta = "/content/drive/My Drive/Colab Notebooks/Master/1.ETL/entregable3/data/raw/extract_energia_sostenible_WB_SE4ALL.csv"
df_consolidado = pd.read_csv(ruta, sep = ';')

In [5]:
# Verificar la cantidad de registros y el número de columnas del dataframe df_consolidado
df_consolidado.shape

(13939, 24)

Al final se obtienen 13939 registros con 24 columnas, la misma información recuperada en la extracción de datos.

In [6]:
# Conocer nombres de las columnas, cantidad de registros y tipos de datos del dataframe df_consolidado
df_consolidado.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13939 entries, 0 to 13938
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   OBS_VALUE         13939 non-null  float64
 1   TIME_FORMAT       13939 non-null  int64  
 2   UNIT_MULT         13939 non-null  int64  
 3   COMMENT_OBS       0 non-null      float64
 4   OBS_STATUS        13939 non-null  object 
 5   OBS_CONF          13939 non-null  object 
 6   AGG_METHOD        13939 non-null  object 
 7   DECIMALS          0 non-null      float64
 8   COMMENT_TS        0 non-null      float64
 9   DATA_SOURCE       0 non-null      float64
 10  LATEST_DATA       13939 non-null  bool   
 11  DATABASE_ID       13939 non-null  object 
 12  INDICATOR         13939 non-null  object 
 13  REF_AREA          13939 non-null  object 
 14  SEX               13939 non-null  object 
 15  AGE               13939 non-null  object 
 16  URBANISATION      13939 non-null  object

Se observa que la columna UNIT_MULT es de tipo entero, la columna LATEST_DATA es de tipo booleano y el resto de columnas son de tipo object. También se observa que las siguientes columnas no cuentan con registros:


*   COMMENT_OBS
*   DECIMALS
*   COMMENT_TS
*   DATA_SOURCE
*   UNIT_TYPE

Se procede a actuar de acuerdo de acuerdos a los hallazgos clave del EDA en la fase de Extracción y a realizar las transformaciones descritas en la segunda entrega de este proyecto.

# Eliminación de columnas

Las transformaciones realizadas se van a guardar en el dataframe df_transform.

Primero se procede a eliminar las columnas que no cuentan con registros:


*   COMMENT_OBS
*   DECIMALS
*   COMMENT_TS
*   DATA_SOURCE
*   UNIT_TYPE

In [7]:
# Elimina las columnas: COMMENT_OBS, DECIMALS, COMMENT_TS, DATA_SOURCE y UNIT_TYPE
# axis = 1 signifca que se va a trabajar con columnas
df_transform = df_consolidado.dropna(axis = 1, how = "all")

In [8]:
df_transform.nunique()

,0
OBS_VALUE,3019
TIME_FORMAT,1
UNIT_MULT,2
OBS_STATUS,1
OBS_CONF,1
AGG_METHOD,2
LATEST_DATA,2
DATABASE_ID,1
INDICATOR,8
REF_AREA,20


Se observa que las siguientes columnas no están en el dataframe df_transform:


*   COMMENT_OBS
*   DECIMALS
*   COMMENT_TS
*   DATA_SOURCE
*   UNIT_TYPE

Recordar que las columnas que tienen un solo valor constante se pueden eliminar siempre y cuando se consideren como valores constantes en el desarrollo y transcurso del proyecto. Estas columnas son:



*   TIME_FORMAT: 602 (el valor del campo de tiempo o fecha debe expresarse únicamente como un año calendario de 4 dígitos )
*   OBS_STATUS: A (valor normal)
*   OBS_CONF: PU (público)
*   DATABASE_ID: WB_SE4ALL (identificador de la base de datos)
*   SEX: _T (total)
*   AGE: _T (todos los rangos de edad o sin desglose por edad)
*   COMP_BREAKDOWN_2: _Z (no aplica)
*   COMP_BREAKDOWN_3: _Z (no aplica)
*   FREQ: A (anual)

In [9]:
columnas_eliminar = [
    "TIME_FORMAT",
    "OBS_STATUS",
    "OBS_CONF",
    "DATABASE_ID",
    "SEX",
    "AGE",
    "COMP_BREAKDOWN_2",
    "COMP_BREAKDOWN_3",
    "FREQ"
]

df_transform = df_transform.drop(columns = columnas_eliminar)

In [10]:
df_transform.head()

,OBS_VALUE,UNIT_MULT,AGG_METHOD,LATEST_DATA,INDICATOR,REF_AREA,URBANISATION,COMP_BREAKDOWN_1,TIME_PERIOD,UNIT_MEASURE
0,49.8,0,W_AVG,False,WB_SE4ALL_EG_FEC_RNEW,BRA,_T,WB_SE4ALL_TYPE_RENEWBLS,1990,PT
1,41.4,0,W_AVG,False,WB_SE4ALL_EG_FEC_RNEW,BRA,_T,WB_SE4ALL_TYPE_MDRNRENWBLS,1990,PT
2,48.9,0,W_AVG,False,WB_SE4ALL_EG_FEC_RNEW,BRA,_T,WB_SE4ALL_TYPE_RENEWBLS,1991,PT
3,40.7,0,W_AVG,False,WB_SE4ALL_EG_FEC_RNEW,BRA,_T,WB_SE4ALL_TYPE_MDRNRENWBLS,1991,PT
4,48.7,0,W_AVG,False,WB_SE4ALL_EG_FEC_RNEW,BRA,_T,WB_SE4ALL_TYPE_RENEWBLS,1992,PT


Luego de la eliminación de las columnas, se observa que el dataframe df_transform queda con 10 columnas que son:


*   OBS_VALUE
*   UNIT_MULT
*   AGG_METHOD
*   LATEST_DATA
*   INDICATOR
*   REF_AREA
*   URBANISATION
*   COMP_BREAKDOWN_1
*   TIME_PERIOD
*   UNIT_MEASURE

# Convertir cadena a número

Convertir las columnas OBS_VALUE y TIME_PERIOD a número, ya que representan el valor de la muestra y el año respectivamente.

In [11]:
# errors = "coerce" indica que si no puede convertir un valor a número, entonces lo convierta a NaN
# Convertir las columnas OBS_VALUE y TIME_PERIOD a número
df_transform["OBS_VALUE"] = pd.to_numeric(df_transform["OBS_VALUE"], errors = "coerce")
df_transform["TIME_PERIOD"] = pd.to_numeric(df_transform["TIME_PERIOD"], errors = "coerce")

Se ajusta los valores de la columna REF_AREA para que no tengan el código del país, sino que tengan la descripción

In [12]:
paises = {
    "BRA": "Brasil",
    "MEX": "México",
    "ARG": "Argentina",
    "CHL": "Chile",
    "COL": "Colombia",
    "PER": "Perú",
    "VEN": "Venezuela",
    "ECU": "Ecuador",
    "DOM": "República Dominicana",
    "PRY": "Paraguay",
    "URY": "Uruguay",
    "GTM": "Guatemala",
    "CRI": "Costa Rica",
    "PAN": "Panamá",
    "BOL": "Bolivia",
    "HND": "Honduras",
    "SLV": "El Salvador",
    "NIC": "Nicaragua",
    "CUB": "Cuba",
    "HTI": "Haití"
}

# fillna significa que si no hay coincidencias con el diccionario de paises, entonces mantenga el código
df_transform['REF_AREA'] = (df_transform['REF_AREA'].map(paises).fillna(df_transform['REF_AREA']))

In [13]:
df_transform.head()

,OBS_VALUE,UNIT_MULT,AGG_METHOD,LATEST_DATA,INDICATOR,REF_AREA,URBANISATION,COMP_BREAKDOWN_1,TIME_PERIOD,UNIT_MEASURE
0,49.8,0,W_AVG,False,WB_SE4ALL_EG_FEC_RNEW,Brasil,_T,WB_SE4ALL_TYPE_RENEWBLS,1990,PT
1,41.4,0,W_AVG,False,WB_SE4ALL_EG_FEC_RNEW,Brasil,_T,WB_SE4ALL_TYPE_MDRNRENWBLS,1990,PT
2,48.9,0,W_AVG,False,WB_SE4ALL_EG_FEC_RNEW,Brasil,_T,WB_SE4ALL_TYPE_RENEWBLS,1991,PT
3,40.7,0,W_AVG,False,WB_SE4ALL_EG_FEC_RNEW,Brasil,_T,WB_SE4ALL_TYPE_MDRNRENWBLS,1991,PT
4,48.7,0,W_AVG,False,WB_SE4ALL_EG_FEC_RNEW,Brasil,_T,WB_SE4ALL_TYPE_RENEWBLS,1992,PT


# Rejilla temporal común

Obtener los registros únicamente entre los años 2000 y 2021

In [14]:
# Obtener los datos desde el 2000 hasta el 2021
df_transform = df_transform[df_transform["TIME_PERIOD"].between(2000, 2021)]

In [15]:
# Analizar los registros de URBANISATION
df_counts = df_transform['URBANISATION'].value_counts().to_frame(name = 'Cantidad')
df_counts['Porcentaje'] = (df_transform['URBANISATION'].value_counts(normalize = True) * 100).round(2)
df_counts

,Cantidad,Porcentaje
URBANISATION,,
_T,10560,85.76
URB,880,7.15
RUR,874,7.10


La mayoría de los datos de la columna URBANISATION están concentrados en el valor _T (Total) con aproximadamente el 86% y el objetivo del estudio es analizar los indicadores de cada país por cada año y no busca analizar brechas o desigualdades entre la urbanización urbana y rural.

Los registros URB y RUR corresponden a desagregaciones territoriales del mismo indicador y al incorporarlos, generarían múltiples observaciones para una misma combinación de país, año e indicador lo que dificultaría la construcción de una estructura de datos país-año y podría producir duplicidades al realizar la integración con la otra fuente de datos.

Por tanto, se seleccionó el valor _T como medida representativa del conjunto de la población del país, manteniendo la coherencia entre el nivel de agregación de los indicadores y la unidad de análisis del estudio.

In [16]:
# Obtener los registros donde URBANISATION sea igual a _T
df_transform = df_transform[df_transform['URBANISATION'] == '_T'].copy()

In [17]:
df_transform.shape

(10560, 10)

In [18]:
df_transform.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10560 entries, 20 to 13937
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   OBS_VALUE         10560 non-null  float64
 1   UNIT_MULT         10560 non-null  int64  
 2   AGG_METHOD        10560 non-null  object 
 3   LATEST_DATA       10560 non-null  bool   
 4   INDICATOR         10560 non-null  object 
 5   REF_AREA          10560 non-null  object 
 6   URBANISATION      10560 non-null  object 
 7   COMP_BREAKDOWN_1  10560 non-null  object 
 8   TIME_PERIOD       10560 non-null  int64  
 9   UNIT_MEASURE      10560 non-null  object 
dtypes: bool(1), float64(1), int64(2), object(6)
memory usage: 835.3+ KB


Al final el dataframe df_transform queda con 10560 registros con 10 columnas, sin valores faltantes.

# Variables derivadas

De acuerdo a la descripción de la columna UNIT_MULT: Exponente en base 10 especificado de modo que al multiplicar los valores numéricos de la observación por 10^UNIT_MULT, se obtiene un valor expresado en la unidad de medida (https://data360.worldbank.org/en/dataset/WB_SE4ALL), se procede a crear la nueva columna OBS_VALUE_AJUSTADO

In [19]:
# Calcular el valor ajustado por el multiplicador
df_transform["OBS_VALUE_AJUSTADO"] = (df_transform["OBS_VALUE"] * (10 ** df_transform["UNIT_MULT"]))

In [20]:
df_transform.head()

,OBS_VALUE,UNIT_MULT,AGG_METHOD,LATEST_DATA,INDICATOR,REF_AREA,URBANISATION,COMP_BREAKDOWN_1,TIME_PERIOD,UNIT_MEASURE,OBS_VALUE_AJUSTADO
20,42.7,0,W_AVG,False,WB_SE4ALL_EG_FEC_RNEW,Brasil,_T,WB_SE4ALL_TYPE_RENEWBLS,2000,PT,42.7
21,37.7,0,W_AVG,False,WB_SE4ALL_EG_FEC_RNEW,Brasil,_T,WB_SE4ALL_TYPE_MDRNRENWBLS,2000,PT,37.7
22,41.3,0,W_AVG,False,WB_SE4ALL_EG_FEC_RNEW,Brasil,_T,WB_SE4ALL_TYPE_RENEWBLS,2001,PT,41.3
23,36.2,0,W_AVG,False,WB_SE4ALL_EG_FEC_RNEW,Brasil,_T,WB_SE4ALL_TYPE_MDRNRENWBLS,2001,PT,36.2
24,42.9,0,W_AVG,False,WB_SE4ALL_EG_FEC_RNEW,Brasil,_T,WB_SE4ALL_TYPE_RENEWBLS,2002,PT,42.9


# Reestructuración

A continuación se hace una reestructuración de la información, donde se quiere obtener filas cuya unicidad sea REF_AREA (país) y TIME_PERIOD (año) y las columnas sean los indicadores con sus respectivas especificaciones brindadas por la columna COMP_BREAKDOWN_1 y tenga como valor el campo OBS_VALUE_AJUSTADO

Ahora, se analizará los valores que contiene la columna COMP_BREAKDOWN_1 que representa la especificación de cada indicador

In [21]:
# Valores diferente que tiene la columna COMP_BREAKDOWN_1
df_transform['COMP_BREAKDOWN_1'].unique()

array(['WB_SE4ALL_TYPE_RENEWBLS', 'WB_SE4ALL_TYPE_MDRNRENWBLS', '_Z',
       'WB_SE4ALL_RENEWSRC_ALLRENEW', 'WB_SE4ALL_RENEWSRC_BIOENRGY',
       'WB_SE4ALL_RENEWSRC_GEOTHRMLENRGY', 'WB_SE4ALL_RENEWSRC_HYDPWR',
       'WB_SE4ALL_RENEWSRC_MRERGY', 'WB_SE4ALL_RENEWSRC_MLTREWBLS',
       'WB_SE4ALL_RENEWSRC_SLRERGY', 'WB_SE4ALL_RENEWSRC_WNDERGY',
       'WB_SE4ALL_TYPE_HEATRSNG', 'WB_SE4ALL_TYPE_TRNSPRT',
       'WB_SE4ALL_TYPE_ELEC_CONSP'], dtype=object)

Se observa que la columna COMP_BREAKDOWN_1 tiene catorce valores diferentes que son:


*   WB_SE4ALL_TYPE_RENEWBLS = Tipo: Renovable
*   WB_SE4ALL_TYPE_MDRNRENWBLS = Tipo: Energías renovables modernas
*   WB_SE4ALL_RENEWSRC_ALLRENEW = Fuente renovable: Todas las renovables
*   WB_SE4ALL_RENEWSRC_BIOENRGY = Fuente renovable: Bioenergía
*   WB_SE4ALL_RENEWSRC_GEOTHRMLENRGY = Fuente renovable: Energía geotérmica
*   WB_SE4ALL_RENEWSRC_HYDPWR = Fuente renovable: Energía hidroeléctrica
*   WB_SE4ALL_RENEWSRC_MRERGY = Fuente renovable: Energía marina
*   WB_SE4ALL_RENEWSRC_MLTREWBLS = Fuente renovable: Múltiples fuentes renovables
*   WB_SE4ALL_RENEWSRC_SLRERGY = Fuente renovable: Energía solar
*   WB_SE4ALL_RENEWSRC_WNDERGY = Fuente renovable: Energía eólica
*   WB_SE4ALL_TYPE_HEATRSNG = Tipo: Aumento de calor
*   WB_SE4ALL_TYPE_TRNSPRT = Tipo: Transporte
*   WB_SE4ALL_TYPE_ELEC_CONSP = Tipo: Consumo de electricidad
*   _Z = No aplica

In [22]:
# Se procede crear una nueva columna para almacenar el indicador con su especificidad, seperados con un guión
df_transform['INDICADOR'] = df_transform['INDICATOR'] + "-" + df_transform['COMP_BREAKDOWN_1']

In [23]:
# Realizar el pivote
df_pivote = df_transform.pivot_table(
    index = ['REF_AREA', 'TIME_PERIOD'],
    columns = 'INDICADOR',
    values = 'OBS_VALUE_AJUSTADO',
    aggfunc = 'first'
).reset_index()

In [24]:
df_pivote.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 440 entries, 0 to 439
Data columns (total 26 columns):
 #   Column                                                   Non-Null Count  Dtype  
---  ------                                                   --------------  -----  
 0   REF_AREA                                                 440 non-null    object 
 1   TIME_PERIOD                                              440 non-null    int64  
 2   WB_SE4ALL_EG_ACS_ELEC-_Z                                 440 non-null    float64
 3   WB_SE4ALL_EG_EGEN_RNEW-WB_SE4ALL_RENEWSRC_ALLRENEW       440 non-null    float64
 4   WB_SE4ALL_EG_EGEN_RNEW-WB_SE4ALL_RENEWSRC_BIOENRGY       440 non-null    float64
 5   WB_SE4ALL_EG_EGEN_RNEW-WB_SE4ALL_RENEWSRC_GEOTHRMLENRGY  440 non-null    float64
 6   WB_SE4ALL_EG_EGEN_RNEW-WB_SE4ALL_RENEWSRC_HYDPWR         440 non-null    float64
 7   WB_SE4ALL_EG_EGEN_RNEW-WB_SE4ALL_RENEWSRC_MRERGY         440 non-null    float64
 8   WB_SE4ALL_EG_EGEN_RNEW-WB_SE4A

In [25]:
df_pivote.shape

(440, 26)

Luego de realizar el pivote, se obtienen 440 registros representando los 20 paises en estudio en los 20 años desde el 2000 hasta el 2021, con 26 columnas que representan la información de los indicadores con sus especificidades.

Se observa que ahora las columnas tienen el nombre de códigos proporcionados por el World Bank Group que no son muy dicientes o son díficiles de interpretar, por esta razón se hace un renombramiento de las columnas finalizando con la unidad de medida

In [26]:
nuevos_nombres_columnas = {
    "REF_AREA": "pais",
    "TIME_PERIOD": "anio",

    "WB_SE4ALL_EG_ACS_ELEC-_Z": "acceso_electricidad_pt",

    "WB_SE4ALL_EG_EGEN_RNEW-WB_SE4ALL_RENEWSRC_ALLRENEW": "capacidad_renovable_total_w_persona",
    "WB_SE4ALL_EG_EGEN_RNEW-WB_SE4ALL_RENEWSRC_BIOENRGY": "capacidad_renovable_bioenergia_w_persona",
    "WB_SE4ALL_EG_EGEN_RNEW-WB_SE4ALL_RENEWSRC_GEOTHRMLENRGY": "capacidad_renovable_geotermica_w_persona",
    "WB_SE4ALL_EG_EGEN_RNEW-WB_SE4ALL_RENEWSRC_HYDPWR": "capacidad_renovable_hidroelectrica_w_persona",
    "WB_SE4ALL_EG_EGEN_RNEW-WB_SE4ALL_RENEWSRC_MRERGY": "capacidad_renovable_marina_w_persona",
    "WB_SE4ALL_EG_EGEN_RNEW-WB_SE4ALL_RENEWSRC_SLRERGY": "capacidad_renovable_solar_w_persona",
    "WB_SE4ALL_EG_EGEN_RNEW-WB_SE4ALL_RENEWSRC_WNDERGY": "capacidad_renovable_eolica_w_persona",

    "WB_SE4ALL_EG_EGY_CLEAN-_Z": "acceso_combustibles_tecnologias_limpias_pt",

    "WB_SE4ALL_EG_FCON_RNEW-WB_SE4ALL_TYPE_ELEC_CONSP": "consumo_renovable_electricidad_pj",
    "WB_SE4ALL_EG_FCON_RNEW-WB_SE4ALL_TYPE_HEATRSNG": "consumo_renovable_aumento_calor_pj",
    "WB_SE4ALL_EG_FCON_RNEW-WB_SE4ALL_TYPE_TRNSPRT": "consumo_renovable_transporte_pj",

    "WB_SE4ALL_EG_FEC_RNEW-WB_SE4ALL_TYPE_MDRNRENWBLS": "proporcion_energias_renovables_modernas_pt",
    "WB_SE4ALL_EG_FEC_RNEW-WB_SE4ALL_TYPE_RENEWBLS": "proporcion_energias_renovables_total_pt",

    "WB_SE4ALL_EG_FEC_TOT-_Z": "consumo_final_total_energia_pj",

    "WB_SE4ALL_EG_INFLW_IC-WB_SE4ALL_RENEWSRC_ALLRENEW": "flujos_financieros_renovables_total_usd",
    "WB_SE4ALL_EG_INFLW_IC-WB_SE4ALL_RENEWSRC_BIOENRGY": "flujos_financieros_bioenergia_usd",
    "WB_SE4ALL_EG_INFLW_IC-WB_SE4ALL_RENEWSRC_GEOTHRMLENRGY": "flujos_financieros_geotermica_usd",
    "WB_SE4ALL_EG_INFLW_IC-WB_SE4ALL_RENEWSRC_HYDPWR": "flujos_financieros_hidroelectrica_usd",
    "WB_SE4ALL_EG_INFLW_IC-WB_SE4ALL_RENEWSRC_MLTREWBLS": "flujos_financieros_multiples_renovables_usd",
    "WB_SE4ALL_EG_INFLW_IC-WB_SE4ALL_RENEWSRC_MRERGY": "flujos_financieros_marina_usd",
    "WB_SE4ALL_EG_INFLW_IC-WB_SE4ALL_RENEWSRC_SLRERGY": "flujos_financieros_solar_usd",
    "WB_SE4ALL_EG_INFLW_IC-WB_SE4ALL_RENEWSRC_WNDERGY": "flujos_financieros_eolica_usd",

    "WB_SE4ALL_EG_INT_LVL-_Z": "intensidad_energetica_mj_pib"
}

df_pivote = df_pivote.rename(columns = nuevos_nombres_columnas)

In [27]:
df_pivote.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 440 entries, 0 to 439
Data columns (total 26 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   pais                                          440 non-null    object 
 1   anio                                          440 non-null    int64  
 2   acceso_electricidad_pt                        440 non-null    float64
 3   capacidad_renovable_total_w_persona           440 non-null    float64
 4   capacidad_renovable_bioenergia_w_persona      440 non-null    float64
 5   capacidad_renovable_geotermica_w_persona      440 non-null    float64
 6   capacidad_renovable_hidroelectrica_w_persona  440 non-null    float64
 7   capacidad_renovable_marina_w_persona          440 non-null    float64
 8   capacidad_renovable_solar_w_persona           440 non-null    float64
 9   capacidad_renovable_eolica_w_persona          440 non-null    flo

# Salida del script
Se guardarán los datos transformados en un csv, para la fase de carga que se trabajará en otro script.

In [28]:
ruta = "/content/drive/My Drive/Colab Notebooks/Master/1.ETL/entregable3/data/transformed/"

# index = False; evita guardar el índice de pandas como una columna adicional
df_pivote.to_csv(ruta + "transform_energia_sostenible_WB_SE4ALL.csv", sep = ";", index = False)